# PODCAST Benchmark — Result Matrix (0423)

Loads foundation-model results for **subject_full** and **persubject_concat** only.
This version checks **lag 0** and **lag 250** and reports missing subjects / folds / lag rows.
No baseline results are included.

In [21]:
import os
import re
import yaml
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

REPO_ROOT = Path(os.path.abspath('')).parent
FM_RESULTS_DIR = REPO_ROOT / 'results' / 'foundation_models'
OUTPUT_DIR = Path(os.path.abspath('')) / 'results_0423'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TASK_DIR_TO_NAME = {
    'content_noncontent': 'content_noncontent_task',
    'sentence_onset': 'sentence_onset_task',
    'iu_boundary': 'iu_boundary_task',
    'gpt_surprise': 'gpt_surprise_task',
    'gpt_surprise_multiclass': 'gpt_surprise_multiclass_task',
    'pos': 'pos_task',
    'word_embedding': 'word_embedding_decoding_task',
    'whisper_embedding': 'whisper_embedding_decoding_task',
    #'volume_level': 'volume_level_decoding_task',
    #'llm_decoding': 'llm_decoding_task',
    'llm_embedding_pretraining': 'llm_embedding_pretraining_task',
}

PRIMARY_METRIC = {
    'content_noncontent_task': 'roc_auc',
    'sentence_onset_task': 'roc_auc',
    'iu_boundary_task': 'roc_auc',
    'gpt_surprise_task': 'corr',
    'gpt_surprise_multiclass_task': 'roc_auc_multiclass',
    'pos_task': 'roc_auc_multiclass',
    'word_embedding_decoding_task': 'word_avg_auc_roc',
    'whisper_embedding_decoding_task': 'pairwise_accuracy',
    'volume_level_decoding_task': 'corr',
    'llm_decoding_task': 'perplexity_llm',
    'llm_embedding_pretraining_task': 'mse',
}

FOUNDATION_MODELS = ['brainbert', 'diver', 'popt']
SUBJECTS = [i for i in range(1, 10) if i != 2]
TARGET_LAGS = [0, 250]
EXPECTED_FOLDS = [1, 2, 3, 4, 5]
ALL_TASKS = sorted(PRIMARY_METRIC.keys())
TASK_SHORT = {t: t.replace('_task', '').replace('_decoding', '') for t in ALL_TASKS}

print(f'Repo root:  {REPO_ROOT}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'FM models:  {FOUNDATION_MODELS}')
print(f'Subjects:   {SUBJECTS}')
print(f'Target lags: {TARGET_LAGS}')

Repo root:  /pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcast-benchmark
Output dir: /pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcast-benchmark/notebooks/results_0423
FM models:  ['brainbert', 'diver', 'popt']
Subjects:   [1, 3, 4, 5, 6, 7, 8, 9]
Target lags: [0, 250]


## Helper Functions

In [22]:
def is_valid_run(run_dir, required_epochs=100):
    cfg_path = Path(run_dir) / 'config.yml'
    if not cfg_path.exists():
        return False
    try:
        with open(cfg_path) as f:
            cfg = yaml.safe_load(f)
        return cfg.get('training_params', {}).get('epochs') == required_epochs
    except Exception:
        return False


def list_valid_runs(condition_dir):
    if not condition_dir.exists():
        return []
    valid = []
    for d in condition_dir.iterdir():
        if not d.is_dir():
            continue
        if not is_valid_run(d):
            continue
        if not (d / 'lag_performance.csv').exists():
            continue
        valid.append((os.path.getmtime(d), d))
    valid.sort(reverse=True)
    return [d for _, d in valid]


def read_lag_metrics(csv_path, task_name, target_lags=TARGET_LAGS):
    df = pd.read_csv(csv_path)
    primary = PRIMARY_METRIC.get(task_name)
    if primary is None:
        return {}

    mean_col = f'test_{primary}_mean'
    std_col = f'test_{primary}_std'
    results = {}

    for lag in target_lags:
        lag_df = df[df['lags'] == lag] if 'lags' in df.columns else pd.DataFrame()
        if lag_df.empty:
            results[lag] = {
                'mean': np.nan,
                'std': np.nan,
                'available_folds': [],
                'missing_folds': EXPECTED_FOLDS,
                'reason': 'missing lag rows',
            }
            continue

        row = lag_df.iloc[0]
        available_folds = []
        for fold in EXPECTED_FOLDS:
            fold_col = f'test_{primary}_fold_{fold}'
            if fold_col in lag_df.columns and pd.notna(row[fold_col]):
                available_folds.append(fold)
        missing_folds = [fold for fold in EXPECTED_FOLDS if fold not in available_folds]

        results[lag] = {
            'mean': row[mean_col] if mean_col in lag_df.columns else np.nan,
            'std': row[std_col] if std_col in lag_df.columns else np.nan,
            'available_folds': available_folds,
            'missing_folds': missing_folds,
            'reason': 'missing folds' if missing_folds else '',
        }

    return results


def select_best_lag_results(condition_dir, task_name, target_lags=TARGET_LAGS):
    run_dirs = list_valid_runs(condition_dir)
    selected = {lag: None for lag in target_lags}
    lag_report = []

    if not run_dirs:
        return selected, [{
            'lag': 'all',
            'available_folds': '',
            'missing_folds': '',
            'reason': 'run directory exists but no valid run found',
        }]

    # Prefer the newest complete run for each lag.
    # If no complete run exists for a lag, fall back to the newest run that has that lag.
    for lag in target_lags:
        best_incomplete = None
        for run in run_dirs:
            run_results = read_lag_metrics(run / 'lag_performance.csv', task_name, target_lags=[lag])
            info = run_results.get(lag)
            if info is None:
                continue
            candidate = {
                'run': run,
                'mean': info['mean'],
                'std': info['std'],
                'available_folds': info['available_folds'],
                'missing_folds': info['missing_folds'],
                'reason': info['reason'],
            }
            if not info['missing_folds']:
                selected[lag] = candidate
                break
            if best_incomplete is None:
                best_incomplete = candidate
        if selected[lag] is None:
            selected[lag] = best_incomplete

        info = selected[lag]
        if info is None:
            lag_report.append({
                'lag': lag,
                'available_folds': '',
                'missing_folds': ','.join(str(x) for x in EXPECTED_FOLDS),
                'reason': 'missing lag rows',
            })
        elif info['missing_folds']:
            lag_report.append({
                'lag': lag,
                'available_folds': ','.join(str(x) for x in info['available_folds']),
                'missing_folds': ','.join(str(x) for x in info['missing_folds']),
                'reason': info['reason'],
            })

    return selected, lag_report


def metric_frame(index_names, columns=ALL_TASKS):
    return pd.DataFrame(index=index_names, columns=columns, dtype=float)


print('Helper functions defined.')

Helper functions defined.


## Load FM Results

In [23]:
fm = {
    'subject_full': {lag: {s: defaultdict(dict) for s in SUBJECTS} for lag in TARGET_LAGS},
    'persubject_concat': {lag: defaultdict(dict) for lag in TARGET_LAGS},
}

missing_report = []

for model in FOUNDATION_MODELS:
    for task_dir, task_name in TASK_DIR_TO_NAME.items():
        # persubject_concat: select best run per lag across all valid runs
        cond_dir = FM_RESULTS_DIR / model / task_dir / 'persubject_concat'
        selected, lag_reports = select_best_lag_results(cond_dir, task_name)
        for lag in TARGET_LAGS:
            info = selected.get(lag)
            if info is None:
                fm['persubject_concat'][lag][model][task_name] = (np.nan, np.nan)
                continue
            fm['persubject_concat'][lag][model][task_name] = (info['mean'], info['std'])
        for item in lag_reports:
            missing_report.append({
                'condition': 'persubject_concat',
                'model': model,
                'task': task_name,
                'subject': 'all',
                'lag': item['lag'],
                'available_folds': item['available_folds'],
                'missing_folds': item['missing_folds'],
                'reason': item['reason'],
            })

        # subject_full: select best run per lag for each subject independently
        for subj in SUBJECTS:
            cond_dir = FM_RESULTS_DIR / model / task_dir / 'subject_full' / f'subject{subj}_full'
            selected, lag_reports = select_best_lag_results(cond_dir, task_name)
            for lag in TARGET_LAGS:
                info = selected.get(lag)
                if info is None:
                    fm['subject_full'][lag][subj][model][task_name] = (np.nan, np.nan)
                    continue
                fm['subject_full'][lag][subj][model][task_name] = (info['mean'], info['std'])
            for item in lag_reports:
                missing_report.append({
                    'condition': 'subject_full',
                    'model': model,
                    'task': task_name,
                    'subject': subj,
                    'lag': item['lag'],
                    'available_folds': item['available_folds'],
                    'missing_folds': item['missing_folds'],
                    'reason': item['reason'],
                })

for lag in TARGET_LAGS:
    print(f'=== Loaded lag {lag} ===')
    for subj in SUBJECTS:
        counts = {m: len(fm['subject_full'][lag][subj].get(m, {})) for m in FOUNDATION_MODELS}
        print(f'subject_full sub{subj}: {counts}')
    counts = {m: len(fm['persubject_concat'][lag].get(m, {})) for m in FOUNDATION_MODELS}
    print(f'persubject_concat: {counts}')

missing_df = pd.DataFrame(missing_report)
if not missing_df.empty:
    missing_df = missing_df.drop_duplicates()
    display(missing_df)
    missing_df.to_csv(OUTPUT_DIR / 'missing_report.csv', index=False)
else:
    print('No missing runs / folds detected.')

=== Loaded lag 0 ===
subject_full sub1: {'brainbert': 9, 'diver': 9, 'popt': 9}
subject_full sub3: {'brainbert': 9, 'diver': 9, 'popt': 9}
subject_full sub4: {'brainbert': 9, 'diver': 9, 'popt': 9}
subject_full sub5: {'brainbert': 9, 'diver': 9, 'popt': 9}
subject_full sub6: {'brainbert': 9, 'diver': 9, 'popt': 9}
subject_full sub7: {'brainbert': 9, 'diver': 9, 'popt': 9}
subject_full sub8: {'brainbert': 9, 'diver': 9, 'popt': 9}
subject_full sub9: {'brainbert': 9, 'diver': 9, 'popt': 9}
persubject_concat: {'brainbert': 9, 'diver': 9, 'popt': 9}
=== Loaded lag 250 ===
subject_full sub1: {'brainbert': 9, 'diver': 9, 'popt': 9}
subject_full sub3: {'brainbert': 9, 'diver': 9, 'popt': 9}
subject_full sub4: {'brainbert': 9, 'diver': 9, 'popt': 9}
subject_full sub5: {'brainbert': 9, 'diver': 9, 'popt': 9}
subject_full sub6: {'brainbert': 9, 'diver': 9, 'popt': 9}
subject_full sub7: {'brainbert': 9, 'diver': 9, 'popt': 9}
subject_full sub8: {'brainbert': 9, 'diver': 9, 'popt': 9}
subject_full

## Build and Save Matrices

In [24]:
summary_rows = []

for lag in TARGET_LAGS:
    print(f'\n==================== LAG {lag} ====================')

    # subject_full: per-subject matrices and subject-averaged summary
    per_subj_full = {}
    per_subj_full_std = {}
    for subj in SUBJECTS:
        mat_m = metric_frame(FOUNDATION_MODELS)
        mat_s = metric_frame(FOUNDATION_MODELS)
        for model in FOUNDATION_MODELS:
            for task in ALL_TASKS:
                m, s = fm['subject_full'][lag][subj].get(model, {}).get(task, (np.nan, np.nan))
                mat_m.loc[model, task] = m
                mat_s.loc[model, task] = s
        per_subj_full[subj] = mat_m
        per_subj_full_std[subj] = mat_s

    stacked = np.stack([per_subj_full[s].values.astype(float) for s in SUBJECTS], axis=0)
    mean_subj_full = pd.DataFrame(np.nanmean(stacked, axis=0), index=FOUNDATION_MODELS, columns=ALL_TASKS)
    std_subj_full = pd.DataFrame(np.nanstd(stacked, axis=0), index=FOUNDATION_MODELS, columns=ALL_TASKS)
    count_subj_full = pd.DataFrame(np.sum(~np.isnan(stacked), axis=0), index=FOUNDATION_MODELS, columns=ALL_TASKS)

    display_sf = pd.DataFrame(index=FOUNDATION_MODELS, columns=ALL_TASKS, dtype=object)
    for model in FOUNDATION_MODELS:
        for task in ALL_TASKS:
            m = mean_subj_full.loc[model, task]
            s = std_subj_full.loc[model, task]
            n = count_subj_full.loc[model, task]
            display_sf.loc[model, task] = f'{m:.4f} +/- {s:.4f} (n={int(n)})' if pd.notna(m) else '-'

    display_sf.columns = [TASK_SHORT[c] for c in display_sf.columns]
    print('=== Subject_full — subject-averaged ===')
    print('  mean +/- std across subjects (each subject value is fold-averaged over the selected lag)')
    display(display_sf)

    mean_subj_full.to_csv(OUTPUT_DIR / f'fm_subject_full_mean_lag{lag}.csv')
    std_subj_full.to_csv(OUTPUT_DIR / f'fm_subject_full_subject_std_lag{lag}.csv')
    count_subj_full.to_csv(OUTPUT_DIR / f'fm_subject_full_subject_count_lag{lag}.csv')
    for subj in SUBJECTS:
        per_subj_full[subj].to_csv(OUTPUT_DIR / f'fm_subject_full_sub{subj}_mean_lag{lag}.csv')
        per_subj_full_std[subj].to_csv(OUTPUT_DIR / f'fm_subject_full_sub{subj}_fold_std_lag{lag}.csv')

    # persubject_concat: one matrix per lag
    mat_psconcat_mean = metric_frame(FOUNDATION_MODELS)
    mat_psconcat_std = metric_frame(FOUNDATION_MODELS)
    for model in FOUNDATION_MODELS:
        for task in ALL_TASKS:
            m, s = fm['persubject_concat'][lag].get(model, {}).get(task, (np.nan, np.nan))
            mat_psconcat_mean.loc[model, task] = m
            mat_psconcat_std.loc[model, task] = s

    display_psc = pd.DataFrame(index=FOUNDATION_MODELS, columns=ALL_TASKS, dtype=object)
    for model in FOUNDATION_MODELS:
        for task in ALL_TASKS:
            m = mat_psconcat_mean.loc[model, task]
            s = mat_psconcat_std.loc[model, task]
            if pd.notna(m):
                display_psc.loc[model, task] = f'{m:.4f} +/- {s:.4f}' if pd.notna(s) else f'{m:.4f}'
            else:
                display_psc.loc[model, task] = '-'

    display_psc.columns = [TASK_SHORT[c] for c in display_psc.columns]
    print('=== Persubject_concat ===')
    print('  mean +/- std across folds (selected lag only)')
    display(display_psc)

    mat_psconcat_mean.to_csv(OUTPUT_DIR / f'fm_persubject_concat_mean_lag{lag}.csv')
    mat_psconcat_std.to_csv(OUTPUT_DIR / f'fm_persubject_concat_fold_std_lag{lag}.csv')

    for model in FOUNDATION_MODELS:
        for task in ALL_TASKS:
            summary_rows.append({
                'condition': 'subject_full',
                'lag': lag,
                'model': model,
                'task': task,
                'primary_metric': PRIMARY_METRIC[task],
                'mean': mean_subj_full.loc[model, task],
                'std': std_subj_full.loc[model, task],
                'n_subjects': int(count_subj_full.loc[model, task]),
            })
            summary_rows.append({
                'condition': 'persubject_concat',
                'lag': lag,
                'model': model,
                'task': task,
                'primary_metric': PRIMARY_METRIC[task],
                'mean': mat_psconcat_mean.loc[model, task],
                'std': mat_psconcat_std.loc[model, task],
                'n_subjects': '',
            })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_DIR / 'all_conditions_long.csv', index=False)

print('\nSaved files:')
for f in sorted(OUTPUT_DIR.glob('*')):
    size_kb = f.stat().st_size / 1024 if f.is_file() else 0
    print(f'  {f.name:45s}  {size_kb:6.1f} KB')


==================== LAG 0 ====================
=== Subject_full — subject-averaged ===
  mean +/- std across subjects (each subject value is fold-averaged over the selected lag)


/tmp/ipykernel_1450625/2743988885.py:21: RuntimeWarning: Mean of empty slice
  mean_subj_full = pd.DataFrame(np.nanmean(stacked, axis=0), index=FOUNDATION_MODELS, columns=ALL_TASKS)
/global/u1/a/ahhyun/.conda/envs/DIVER/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


,content_noncontent,gpt_surprise_multiclass,gpt_surprise,iu_boundary,llm,llm_embedding_pretraining,pos,sentence_onset,volume_level,whisper_embedding,word_embedding
brainbert,0.5288 +/- 0.0168 (n=8),0.5062 +/- 0.0095 (n=8),0.0076 +/- 0.0127 (n=8),0.6359 +/- 0.0519 (n=8),-,0.0544 +/- 0.0172 (n=8),0.5200 +/- 0.0170 (n=8),0.6088 +/- 0.0456 (n=8),-,0.5608 +/- 0.0230 (n=8),0.5471 +/- 0.0213 (n=8)
diver,0.5533 +/- 0.0315 (n=8),0.5135 +/- 0.0120 (n=8),0.0408 +/- 0.0243 (n=8),0.6842 +/- 0.0779 (n=8),-,0.0113 +/- 0.0001 (n=8),0.5553 +/- 0.0310 (n=8),0.6553 +/- 0.0725 (n=8),-,0.6380 +/- 0.0551 (n=8),0.6119 +/- 0.0471 (n=8)
popt,0.4986 +/- 0.0141 (n=8),0.5023 +/- 0.0049 (n=8),-0.0041 +/- 0.0084 (n=8),0.5144 +/- 0.0147 (n=8),-,0.0115 +/- 0.0001 (n=8),0.4988 +/- 0.0036 (n=8),0.5033 +/- 0.0128 (n=8),-,0.5017 +/- 0.0020 (n=8),0.4951 +/- 0.0108 (n=8)


=== Persubject_concat ===
  mean +/- std across folds (selected lag only)


,content_noncontent,gpt_surprise_multiclass,gpt_surprise,iu_boundary,llm,llm_embedding_pretraining,pos,sentence_onset,volume_level,whisper_embedding,word_embedding
brainbert,0.5833 +/- 0.0084,0.5240 +/- 0.0132,0.0052 +/- 0.0252,0.7427 +/- 0.0327,-,0.1067 +/- 0.0022,0.5599 +/- 0.0132,0.7175 +/- 0.0319,-,0.6824 +/- 0.0148,0.6339 +/- 0.0116
diver,0.6560 +/- 0.0222,0.5476 +/- 0.0200,0.0978 +/- 0.0271,0.7927 +/- 0.0240,-,0.0113 +/- 0.0002,0.6561 +/- 0.0177,0.7940 +/- 0.0038,-,0.7920 +/- 0.0175,0.7477 +/- 0.0107
popt,0.5474 +/- 0.0307,0.5324 +/- 0.0104,0.0400 +/- 0.0233,0.7062 +/- 0.0113,-,0.0133 +/- 0.0002,0.5286 +/- 0.0188,0.6748 +/- 0.0359,-,0.6211 +/- 0.0056,0.5785 +/- 0.0145


...

/tmp/ipykernel_1450625/2743988885.py:21: RuntimeWarning: Mean of empty slice
  mean_subj_full = pd.DataFrame(np.nanmean(stacked, axis=0), index=FOUNDATION_MODELS, columns=ALL_TASKS)
/global/u1/a/ahhyun/.conda/envs/DIVER/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


,content_noncontent,gpt_surprise_multiclass,gpt_surprise,iu_boundary,llm,llm_embedding_pretraining,pos,sentence_onset,volume_level,whisper_embedding,word_embedding
brainbert,0.5273 +/- 0.0131 (n=8),0.5094 +/- 0.0089 (n=8),0.0077 +/- 0.0137 (n=8),0.5856 +/- 0.0333 (n=8),-,0.0548 +/- 0.0177 (n=8),0.5288 +/- 0.0179 (n=8),0.6505 +/- 0.0507 (n=8),-,0.5705 +/- 0.0263 (n=8),0.5581 +/- 0.0221 (n=8)
diver,0.5626 +/- 0.0322 (n=8),0.5117 +/- 0.0130 (n=8),0.0505 +/- 0.0261 (n=8),0.6893 +/- 0.0750 (n=8),-,0.0112 +/- 0.0001 (n=8),0.5695 +/- 0.0342 (n=8),0.6471 +/- 0.0628 (n=8),-,0.6555 +/- 0.0615 (n=8),0.6234 +/- 0.0500 (n=8)
popt,0.5012 +/- 0.0074 (n=8),0.5011 +/- 0.0061 (n=8),0.0034 +/- 0.0135 (n=8),0.5015 +/- 0.0218 (n=8),-,0.0115 +/- 0.0001 (n=8),0.5042 +/- 0.0114 (n=8),0.5077 +/- 0.0143 (n=8),-,0.5020 +/- 0.0035 (n=8),0.5025 +/- 0.0114 (n=8)


=== Persubject_concat ===
  mean +/- std across folds (selected lag only)


,content_noncontent,gpt_surprise_multiclass,gpt_surprise,iu_boundary,llm,llm_embedding_pretraining,pos,sentence_onset,volume_level,whisper_embedding,word_embedding
brainbert,0.5983 +/- 0.0082,0.5173 +/- 0.0117,0.0254 +/- 0.0121,0.7103 +/- 0.0218,-,0.1077 +/- 0.0022,0.6016 +/- 0.0102,0.7525 +/- 0.0333,-,0.7327 +/- 0.0146,0.6629 +/- 0.0165
diver,0.6503 +/- 0.0163,0.5532 +/- 0.0323,0.1092 +/- 0.0146,0.7927 +/- 0.0301,-,0.0113 +/- 0.0001,0.6633 +/- 0.0196,0.7960 +/- 0.0080,-,0.8198 +/- 0.0136,0.7702 +/- 0.0189
popt,0.5426 +/- 0.0255,0.5274 +/- 0.0160,0.0565 +/- 0.0149,0.6692 +/- 0.0171,-,0.0133 +/- 0.0002,0.5453 +/- 0.0058,0.5025 +/- 0.0535,-,0.6241 +/- 0.0085,0.5909 +/- 0.0107



Saved files:
  all_conditions_long.csv                          12.1 KB
  fm_persubject_concat_fold_std_lag0.csv            0.8 KB
  fm_persubject_concat_fold_std_lag250.csv          0.8 KB
  fm_persubject_concat_mean_lag0.csv                0.8 KB
  fm_persubject_concat_mean_lag250.csv              0.8 KB
  fm_subject_full_mean_lag0.csv                     0.8 KB
  fm_subject_full_mean_lag250.csv                   0.8 KB
  fm_subject_full_sub1_fold_std_lag0.csv            0.8 KB
  fm_subject_full_sub1_fold_std_lag250.csv          0.8 KB
  fm_subject_full_sub1_mean_lag0.csv                0.8 KB
  fm_subject_full_sub1_mean_lag250.csv              0.8 KB
  fm_subject_full_sub3_fold_std_lag0.csv            0.8 KB
  fm_subject_full_sub3_fold_std_lag250.csv          0.8 KB
  fm_subject_full_sub3_mean_lag0.csv                0.8 KB
  fm_subject_full_sub3_mean_lag250.csv              0.8 KB
  fm_subject_full_sub4_fold_std_lag0.csv            0.8 KB
  fm_subject_full_sub4_fold_std_lag250.csv